### Concept 1: What abstraction is solving

In [ ]:
class VectorStore:
    def add(self, vector):
        pass   # "meant" to be overridden, but nothing enforces that

    def search(self, query):
        pass   # "meant" to be overridden too

class BrokenVectorStore(VectorStore):
    def add(self, vector):
        print("  added:", vector)
    # forgot to implement search() entirely!

store = BrokenVectorStore()
store.add("some vector")

result = store.search("some query")
print("search result:", result)

BrokenVectorStore is genuinely incomplete — it never implements search() at all. Yet Python happily let you create it (store = BrokenVectorStore()), and store.search("some query") didn't error either — it just silently ran the parent's empty placeholder and returned None. This is a real, dangerous bug: nothing tells you anything is wrong until you notice, potentially much later, that search results are mysteriously always None.

#### Abstraction is about fixing this: defining what a class must be able to do, without saying how — and having the language actually enforce that every subclass fills in the "how."

### Concepts 2 & 3: ABC, @abstractmethod, and what happens if you skip one

What ABC is

ABC stands for Abstract Base Class. It's a special class Python gives you (from the abc module) whose entire job is to say: "classes that inherit from me are templates, not finished objects — you can't create instances of them directly."

class VectorStore(ABC):
you're telling Python: "VectorStore is a template/blueprint. Don't let anyone create a raw VectorStore() object."

What @abstractmethod does

Putting @abstractmethod on add and search marks them as "required to override." It doesn't matter that they have bodies (print("Hi") / return None) — that code never actually matters for the check.

In [ ]:
from abc import ABC, abstractmethod

class VectorStore(ABC):
    @abstractmethod
    def add(self, vector):
        print("Hi")
        return None

    @abstractmethod
    def search(self, query):
        print("Hi")
        return None

vs = VectorStore()

In [ ]:
class BrokenVectorStore(VectorStore):
    def add(self, vector):
        print("  added:", vector)
    # still missing search()!

store = BrokenVectorStore()

This is the actual fix for last concept's bug. BrokenVectorStore still forgets search() — but this time, Python refuses to let the object be created at all, and tells you exactly which method is missing.

A class with any unimplemented @abstractmethod cannot be instantiated — Python raises TypeError at __init__ time.

A subclass becomes instantiable only once it overrides every abstract method inherited from its bases.

If a subclass implements only some of them, it's still abstract itself — you get the same error until all are filled in.

The body of an abstract method (even pass or print(...)) is basically irrelevant — some people put a docstring or raise 

NotImplementedError there as documentation, but Python doesn't run that code unless a subclass explicitly calls super().add(...).